# MOP × DP2: cobertura, calidad y reportes

Notebook reproducible para construir la tabla por target, evaluar visitas/coadds, encontrar matches fotométricos DP2, generar skyplots y producir reportes individuales con cutouts y curvas de luz MOP + DP2.

Primera corrida recomendada: dejar `RUN_COADD_CUTOUTS=False` y `RUN_TARGET_REPORTS=False`; validar tablas y luego activarlos.


In [1]:
from pathlib import Path
import json, sys, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm, Normalize
from IPython.display import display

PROJECT_ROOT = Path('/home/karennowo/target_selection')
if str(PROJECT_ROOT) not in sys.path: sys.path.insert(0, str(PROJECT_ROOT))
from target_selection.collection_access import open_collection, load_collection_coverage
from data_release_config import get_data_release
from target_selection.reference_catalog import ReferenceCatalogConfig, enrich_reference_catalog_coadds
from target_selection.rsp_access import get_coadd_cutouts
from target_selection.tasks.lsst import compute_lsst_photometry
from target_selection.tasks.reports import create_target_reports
from target_selection.reporting import write_collection_summaries
from target_selection.sources.mop import load_mop_targets
from target_selection.quality import apply_coadd_quality_flags
from target_selection.collection_access import query_visit_centers_by_band
from target_selection.plotting import make_skyplots
from dotenv import load_dotenv
load_dotenv("/home/karennowo/target_selection/.env")

True

In [2]:
MOP_ROOT = PROJECT_ROOT / 'outputs' / 'mop_dp2_all'
MOP_TABLE = MOP_ROOT / 'tables' / 'mop_targets_enriched.csv'
MOP_SNAPSHOT = MOP_ROOT / 'raw'
MOP_SOURCE = None
OUT = MOP_ROOT / 'dp2'
CACHE = MOP_ROOT / 'cache' / 'dp2'
MOP_PHOTOMETRY_DIR = MOP_ROOT / 'cache' / 'mop_photometry'
BANDS = tuple('ugrizy')
RUN_VISIT_COVERAGE = True
RUN_COADD_PROPERTIES = True
RUN_DP2_PHOTOMETRY = True
RUN_COADD_CUTOUTS = True
RUN_TARGET_REPORTS = True
RUN_SKYPLOTS = True
FORCE_RECOMPUTE = False
REPORT_LIMIT = None
MAX_WORKERS = 2
OUT.mkdir(parents=True, exist_ok=True); CACHE.mkdir(parents=True, exist_ok=True)
for x in ('tables','summaries','skyplots','target_reports','cutouts'): (OUT/x).mkdir(parents=True, exist_ok=True)


## 1. Cargar y normalizar MOP

Se usa el catálogo enriquecido si existe; si no, el snapshot completo descargado por la API.


In [3]:
# El helper busca primero el catálogo enriquecido y, si no existe, el último export completo en raw/.
targets, mop_source = load_mop_targets(MOP_ROOT, source_path=MOP_SOURCE)
print(f'Fuente MOP: {mop_source}')
targets_valid = targets.loc[targets['coordinates_valid']].copy()
print(f'Objetos cargados: {len(targets):,}')
print(f'Objetos con coordenadas válidas para DP2: {len(targets_valid):,}')
if len(targets_valid) != len(targets): print('Se conservan en el catálogo, pero no se consultan en DP2, los targets sin coordenadas válidas.')
display(targets.head())


Fuente MOP: /home/karennowo/target_selection/outputs/mop_dp2_all/raw/mop_targets_20260923T010143Z.csv
Objetos cargados: 6,445
Objetos con coordenadas válidas para DP2: 6,444
Se conservan en el catálogo, pero no se consultan en DP2, los targets sin coordenadas válidas.


,target_id,Unnamed: 0,Name,Target Type,Right Ascension,Declination,mop_link,Target,ra_deg,dec_deg,RA_deg,Dec_deg,coordinates_valid
0,Gaia19bnd,NaN,Gaia19bnd,Sidereal,336.45009999999996,25.41967,https://mop.lco.global/targets/57353/,Gaia19bnd,336.450100,25.419670,336.450100,25.419670,True
1,ZTF26abbbguw,NaN,ZTF26abbbguw,Sidereal,289.6308951000083,15.31526731,https://mop.lco.global/targets/57320/,ZTF26abbbguw,289.630895,15.315267,289.630895,15.315267,True
2,AT2026aabo,NaN,AT2026aabo,Sidereal,11.226083191645,41.552913459777,https://mop.lco.global/targets/57287/,AT2026aabo,11.226083,41.552913,11.226083,41.552913,True
3,ZTF26abrjfga,NaN,ZTF26abrjfga,Sidereal,271.01420286470295,-8.46602394146538,https://mop.lco.global/targets/57253/,ZTF26abrjfga,271.014203,-8.466024,271.014203,-8.466024,True
4,ZTF26abnuqyj,NaN,ZTF26abnuqyj,Sidereal,114.8408785416924,34.41202727499999,https://mop.lco.global/targets/57220/,ZTF26abnuqyj,114.840879,34.412027,114.840879,34.412027,True


## 2. Cobertura de visitas DP2

`load_collection_coverage` consulta `VisitDetector`, entrega conteos por banda y conserva los matches largos en `coverage.attrs['detector_matches']`.


In [4]:
collection = open_collection('DP2')
release = get_data_release('DP2')
if RUN_VISIT_COVERAGE:
    coverage = load_collection_coverage(targets_valid, collection='DP2', tap_service=collection.tap_service,
        cache_dir=CACHE/'visits', reuse_cache=True, bands=BANDS, max_workers=MAX_WORKERS, verbose=True)
    coverage.to_csv(OUT/'tables'/'dp2_visit_coverage.csv', index=False)
else: coverage = pd.read_csv(OUT/'tables'/'dp2_visit_coverage.csv')
for b in BANDS:
    col=f'n_images_{b}'; coverage[col]=pd.to_numeric(coverage.get(col,0),errors='coerce').fillna(0).astype(int)
    coverage[f'has_valid_visit_{b}']=coverage[col].gt(0)
coverage['has_valid_visit_any']=coverage['n_images_total'].gt(0)
print(f"Targets con visitas: {coverage.has_valid_visit_any.sum():,}/{len(coverage):,}")


Visit tiles: 100%|██████████| 1997/1997 [00:16<00:00, 124.05tile/s]


Visit coverage: 1997 spatial tiles, 6444 targets, 27808 target-band rows
Targets con visitas: 5,184/6,444


In [5]:
detector_matches = coverage.attrs.get('detector_matches', pd.DataFrame()).copy()
if not detector_matches.empty:
    report_visits = detector_matches.rename(columns={'visit_id':'visitId','mjd':'expMidptMJD','mag_lim':'magLim'})
    report_visits = report_visits.merge(targets[['target_id','Target']], on='target_id', how='left')
    if 'detector' not in report_visits: report_visits['detector']=pd.NA
    report_visits.to_csv(OUT/'tables'/'dp2_detector_matches.csv', index=False)
else: report_visits=pd.DataFrame()
print(f'Matches largos: {len(report_visits):,}')


Matches largos: 170,996


## 3. Coadds y calidad

Se separan existencia de dataset, métricas válidas y píxeles válidos. Los píxeles sólo se validan después de leer los cutouts.


In [6]:
COADD_PROPERTIES_PATH=OUT/'tables'/'dp2_coadd_properties.csv'
if COADD_PROPERTIES_PATH.exists() and not FORCE_RECOMPUTE:
    print(f'Reutilizando propiedades de coadds: {COADD_PROPERTIES_PATH}')
    coadd_catalog=pd.read_csv(COADD_PROPERTIES_PATH)
elif RUN_COADD_PROPERTIES:
    cfg=ReferenceCatalogConfig(input_path=mop_source,
        output_dir=OUT/'reference_catalog', cache_dir=CACHE/'coadds', name='mop_dp2_all', data_release='DP2',
        bands=BANDS, target_id_column='target_id', ra_column='ra_deg', dec_column='dec_deg',
        coadd_skymap='lsst_cells_v2', property_maps=('maglim','psf_size','exposure_time','sky_noise'),
        partial_property_map_reads=True, generate_cutouts=False, max_cutouts=0, verbose=True)
    coadd_catalog=enrich_reference_catalog_coadds(cfg, coverage, butler=collection.butler)
    coadd_catalog.to_csv(COADD_PROPERTIES_PATH, index=False)
else:
    raise FileNotFoundError(f'No existe {COADD_PROPERTIES_PATH}; activar RUN_COADD_PROPERTIES=True')
print(len(coadd_catalog)); display(coadd_catalog.head())


Reutilizando propiedades de coadds: /home/karennowo/target_selection/outputs/mop_dp2_all/dp2/tables/dp2_coadd_properties.csv
6444


,target_id,Unnamed: 0,Name,Target Type,Right Ascension,Declination,mop_link,Target,ra_deg,dec_deg,...,coadd_sky_noise_median_njy,selection_grade,selection_grade_source,cutout_grade_rank,cutout_eligible,cutout_priority_rank,cutout_selected,cutout_status,cutout_path,cutout_n_bands
0,Gaia19bnd,NaN,Gaia19bnd,Sidereal,336.450100,25.419670,https://mop.lco.global/targets/57353/,Gaia19bnd,336.450100,25.419670,...,NaN,NaN,unavailable,NaN,False,NaN,False,no_coadd,NaN,0
1,ZTF26abbbguw,NaN,ZTF26abbbguw,Sidereal,289.630895,15.315267,https://mop.lco.global/targets/57320/,ZTF26abbbguw,289.630895,15.315267,...,NaN,NaN,unavailable,NaN,False,NaN,False,no_coadd,NaN,0
2,AT2026aabo,NaN,AT2026aabo,Sidereal,11.226083,41.552913,https://mop.lco.global/targets/57287/,AT2026aabo,11.226083,41.552913,...,NaN,NaN,unavailable,NaN,False,NaN,False,no_coadd,NaN,0
3,ZTF26abrjfga,NaN,ZTF26abrjfga,Sidereal,271.014203,-8.466024,https://mop.lco.global/targets/57253/,ZTF26abrjfga,271.014203,-8.466024,...,NaN,NaN,unavailable,NaN,False,NaN,False,no_coadd,NaN,0
4,ZTF26abnuqyj,NaN,ZTF26abnuqyj,Sidereal,114.840879,34.412027,https://mop.lco.global/targets/57220/,ZTF26abnuqyj,114.840879,34.412027,...,NaN,NaN,unavailable,NaN,False,NaN,False,no_coadd,NaN,0


In [7]:
# Las flags de calidad son genéricas y se reutilizan también en batch.
quality_catalog = apply_coadd_quality_flags(coadd_catalog, bands=BANDS)
quality_catalog.to_csv(OUT/'tables'/'dp2_coadd_quality_pre_cutouts.csv', index=False)


## 4. Cutouts y flags finales

Activar `RUN_COADD_CUTOUTS=True` para validar realmente los píxeles y generar el manifiesto.


In [8]:
cutout_manifest=OUT/'cutouts'/'cutout_manifest.csv'
if RUN_COADD_CUTOUTS:
    # Los cutouts son de 20 arcsec por lado; el manifest se reutiliza directamente. Esto evita
    # borrar/reconstruir el manifest y volver a recorrer todos los targets.
    expected_cutout_targets=int(quality_catalog['has_coadd'].fillna(False).astype(bool).sum())
    existing_manifest=pd.read_csv(cutout_manifest, low_memory=False) if cutout_manifest.exists() else pd.DataFrame()
    manifest_complete=(not existing_manifest.empty and existing_manifest['target_id'].astype(str).nunique() >= expected_cutout_targets)
    if manifest_complete:
        cutout_result=existing_manifest
        print(f'Reutilizando manifest completo: {len(cutout_result):,} registros')
    else:
        cutout_result=get_coadd_cutouts(quality_catalog, collection='DP2', butler=collection.butler,
            output_dir=OUT/'cutouts'/'png', fits_dir=OUT/'cutouts'/'fits', bands=BANDS, size_arcsec=20.0,
            manifest_path=cutout_manifest, reuse_existing=True, verbose=True)
    quality_catalog=apply_coadd_quality_flags(quality_catalog, cutout_result)
else: print('RUN_COADD_CUTOUTS=False: píxeles aún no validados')
quality_catalog.to_csv(OUT/'tables'/'dp2_coadd_quality.csv',index=False)
quality_catalog.loc[quality_catalog.has_valid_coadd].to_csv(OUT/'tables'/'valid_coadd_targets.csv',index=False)
for b in BANDS: print(b, int(quality_catalog[f'has_valid_coadd_{b}'].sum()))


Reutilizando manifest completo: 4,731 registros
u 0
g 0
r 0
i 0
z 0
y 0


In [9]:
try:
    write_collection_summaries(quality_catalog, OUT/'summaries', bands=BANDS,
        category_column='mop_classification' if 'mop_classification' in quality_catalog else None, name='mop_dp2')
except Exception as exc: warnings.warn(str(exc))
rows=[('n_targets',len(quality_catalog)),('n_with_any_visit',int(quality_catalog.has_valid_visit_any.sum())),
      ('n_with_valid_metrics_any',int(quality_catalog.has_valid_metrics_any.sum())),('n_with_valid_pixels_any',int(quality_catalog.has_valid_pixels_any.sum())),
      ('n_with_valid_coadd_any',int(quality_catalog.has_valid_coadd.sum()))]
for b in BANDS: rows += [(f'n_visits_{b}',int(quality_catalog[f'n_images_{b}'].sum())),(f'n_targets_with_visits_{b}',int(quality_catalog[f'has_valid_visit_{b}'].sum())),(f'n_valid_coadds_{b}',int(quality_catalog[f'has_valid_coadd_{b}'].sum()))]
pd.DataFrame(rows,columns=['quantity','value']).to_csv(OUT/'summaries'/'summary_counts.csv',index=False)


## 5. Match con fotometría DIA forced de DP2


In [10]:
DP2_PHOTOMETRY_PATH=OUT/'tables'/'dp2_dia_forced_photometry.csv'
if DP2_PHOTOMETRY_PATH.exists() and not FORCE_RECOMPUTE:
    print(f'Reutilizando fotometría DP2: {DP2_PHOTOMETRY_PATH}')
    dp2_photometry=pd.read_csv(DP2_PHOTOMETRY_PATH)
elif RUN_DP2_PHOTOMETRY:
    dp2_photometry=compute_lsst_photometry(targets_valid, method='dia_forced_catalog', tap_service=collection.tap_service,
        output_path=DP2_PHOTOMETRY_PATH, persistent_target_dir=OUT/'cache'/'dp2_photometry',
        max_workers=MAX_WORKERS, verbose=True)
else:
    raise FileNotFoundError(f'No existe {DP2_PHOTOMETRY_PATH}; activar RUN_DP2_PHOTOMETRY=True')

# La tarea conserva el nombre externo MOP `Target`; usamos una clave de
# agrupación local sin cambiar la tabla fotométrica, porque los reportes
# necesitan conservar sus columnas originales.
if dp2_photometry.empty:
    dp2_match=targets[['target_id']].assign(dp2_dia_match=False,dp2_n_forced_points=0)
else:
    if 'measurement_status' not in dp2_photometry.columns:
        dp2_photometry['measurement_status']='measured'
    phot_target_column='target_id' if 'target_id' in dp2_photometry.columns else 'Target'
    dp2_match=dp2_photometry.groupby(phot_target_column).agg(
        dp2_dia_match=('measurement_status',lambda s:s.eq('measured').any()),
        dp2_n_forced_points=('measurement_status',lambda s:s.eq('measured').sum()),
    ).reset_index().rename(columns={phot_target_column:'target_id'})
print('Matches DIA:',int(dp2_match.dp2_dia_match.sum()))


Reutilizando fotometría DP2: /home/karennowo/target_selection/outputs/mop_dp2_all/dp2/tables/dp2_dia_forced_photometry.csv
Matches DIA: 504


## 6. Skyplots

El fondo gris son los centros de visitas DP2. Los puntos son targets con visitas/coadds válidos y el color representa el conteo.


In [11]:
# Se cachean los centros de visitas para no repetir la consulta TAP.
# Los skyplots no necesitan esperar a la fotometría DIA: si final_catalog
# todavía no existe, usan quality_catalog, que ya contiene RA/Dec, visitas y coadds.
if RUN_SKYPLOTS:
    centers_path = OUT/'tables'/'dp2_visit_centers.csv'
    visit_centers = pd.read_csv(centers_path) if centers_path.exists() else query_visit_centers_by_band(collection.tap_service, release)
    visit_centers.to_csv(centers_path, index=False)
    sky_catalog = final_catalog if 'final_catalog' in globals() else quality_catalog
    make_skyplots(sky_catalog, visit_centers, OUT/'skyplots', bands=BANDS)


/home/karennowo/target_selection/target_selection/mop_dp2.py:122: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout(); fig.savefig(output_dir / "visits_by_band.png", dpi=180); plt.close(fig)


## 7. Reporte individual: cutouts y curva de luz

Para probar, usar `REPORT_LIMIT=10`; para toda la muestra, `REPORT_LIMIT=None`.


In [ ]:
if RUN_TARGET_REPORTS:
    # Los reportes pueden generarse aunque la celda de resumen final haya
    # fallado: quality_catalog ya contiene posiciones y coadds.
    report_catalog = final_catalog if 'final_catalog' in globals() else quality_catalog
    if 'report_visits' not in globals():
        report_visits_path=OUT/'tables'/'dp2_detector_matches.csv'
        report_visits=pd.read_csv(report_visits_path) if report_visits_path.exists() else pd.DataFrame()
    if 'dp2_photometry' not in globals():
        photometry_path=OUT/'tables'/'dp2_dia_forced_photometry.csv'
        dp2_photometry=pd.read_csv(photometry_path) if photometry_path.exists() else pd.DataFrame()
    if 'Target' not in dp2_photometry.columns and 'target_id' in dp2_photometry.columns:
        dp2_photometry['Target']=dp2_photometry['target_id']
    # Usa los CSV MOP cacheados y consulta MOP sólo para targets cuyo cache falte.
    from mop_api import MOPClient
    mop_client = MOPClient()
    rt=report_catalog.loc[report_catalog['has_coadd'].fillna(False)].copy()
    if REPORT_LIMIT is not None: rt=rt.head(REPORT_LIMIT)
    paths=create_target_reports(rt,coverage=report_visits,release_photometry=dp2_photometry,output_dir=OUT/'target_reports',
        mop=mop_client,mop_photometry_dir=MOP_PHOTOMETRY_DIR,data_release='DP2',butler=collection.butler,tap_service=collection.tap_service,
        target_names=rt.Target.tolist(),verbose=True)
    pd.DataFrame({'report_path':paths}).to_csv(OUT/'tables'/'target_report_paths.csv',index=False)
else: print('RUN_TARGET_REPORTS=False: no se generan reportes')


[reports] Creating 4731 target report(s).


Target reports:   0%|          | 0/4731 [00:00<?, ?target/s]

[reports] Building target report: ZTF26aawmspr
[reports] Report saved: /home/karennowo/target_selection/outputs/mop_dp2_all/dp2/target_reports/ZTF26aawmspr_target_report.png
[reports] Building target report: OGLE-2026-BLG-0888
[reports] Report saved: /home/karennowo/target_selection/outputs/mop_dp2_all/dp2/target_reports/OGLE-2026-BLG-0888_target_report.png
[reports] Building target report: ZTF26aauqxam
[reports] Report saved: /home/karennowo/target_selection/outputs/mop_dp2_all/dp2/target_reports/ZTF26aauqxam_target_report.png
[reports] Building target report: LSST_313761042268160096
[reports] Report saved: /home/karennowo/target_selection/outputs/mop_dp2_all/dp2/target_reports/LSST_313761042268160096_target_report.png
[reports] Building target report: ZTF26aazuuhy
[reports] Report saved: /home/karennowo/target_selection/outputs/mop_dp2_all/dp2/target_reports/ZTF26aazuuhy_target_report.png
[reports] Building target report: ZTF26abdgtui
[reports] Report saved: /home/karennowo/target_se

## Productos principales

- `tables/mop_dp2_per_target.csv`: una fila por target con visitas, coadds, flags y match DIA.
- `tables/valid_coadd_targets.csv`: lista de objetos con coadd válido en al menos una banda.
- `summaries/summary_counts.csv` y `summaries/mop_dp2_match_summary.csv`.
- `skyplots/`: mapas solicitados.
- `cutouts/` y `target_reports/`: productos visuales al activar los flags correspondientes.

Para el resultado final de coadds hay que ejecutar primero con `RUN_COADD_CUTOUTS=True`; antes de eso no se considera validado el contenido de píxeles.
